In [17]:
import pandas as pd
import unicodedata
import re

customer = pd.read_csv(r'C:\Users\Ivan.DeLaFuente\OneDrive - Quaker Houghton\Documents\QH WorkDay\Junio 2026\T&P Market\T&P_Customers.csv')
potential = pd.read_csv(r'C:\Users\Ivan.DeLaFuente\OneDrive - Quaker Houghton\Documents\QH WorkDay\Junio 2026\T&P Market\T&P_Potential.csv')
coordinates = pd.read_csv(r'C:\Users\Ivan.DeLaFuente\OneDrive - Quaker Houghton\Documents\QH WorkDay\Junio 2026\T&P Market\Coordinates.csv')
regions = pd.read_csv(r'C:\Users\Ivan.DeLaFuente\OneDrive - Quaker Houghton\Documents\QH WorkDay\Junio 2026\T&P Market\Country_Regions.csv')


### Renombrar columnas para concatenar

In [18]:
customer_rename = {
    "llave": "llave",
    "Global Parent": "Global Parent",
    "GP_country": "GP Country",
    "GP_state": "GP State",
    "GP_city": "GP City",
    "GP_latitude": "GP Latitude",
    "GP_longitude": "GP Longitude",
    "GP_Super region": "GP Super Region",
    "GP_Region": "GP Region",
    "GP_Sub region": "GP Sub Region",
    "ID_customer": "ID Customer",
    "ShipTo Number": "Ship To Number",
    "Customer name": "Customer Name",
    "Channel": "Channel",
    "ShipTo Country": "Ship To Country",
    "ShipTo State": "Ship To State",
    "ShipTo City": "Ship To City",
    "Latitud": "CN Latitude",
    "Longitd": "CN Longitude",
    "Continent": "CN Continent",
    "Super Region": "CN Super Region",
    "Region": "CN Region",
    "Sub Region": "CN Sub Region",
    "Customer Segment Major": "Customer Segment Major",
    "Customer Segment Sub Major": "Customer Segment Sub Major",
    "Customer Segment Minor": "Customer Segment Minor",
    "End Market Major": "End Market Major",
    "End Market Sub Major": "End Market Sub Major",
    "End Market Minor": "End Market Minor",
    "Industry": "Industry",
    "ID_Global parent": "ID_Global parent"
}

In [19]:
potential_rename = {
    "Global Parent" : "Global Parent",
    "GP_country" : "GP Country",
    "GP_state" : "GP State",
    "GP_city" : "GP City",
    "GP_latitude" : "GP Latitude",
    "GP_longitude" : "GP Longitude",
    "GP_Super region" : "GP Super Region",
    "GP_Region" : "GP Region",
    "GP_Sub region" : "GP Sub Region",
    "Customer name" : "Customer Name",
    "ShipTo Country" : "Ship To Country",
    "ShipTo State" : "Ship To State",
    "ShipTo City" : "Ship To City",
    "Latitud" : "CN Latitude",
    "Longitd" : "CN Longitude",
    "Continent" : "CN Continent",
    "Super Region" : "CN Super Region",
    "Region" : "CN Region",
    "Sub Region" : "CN Sub Region",
    "Customer Segment Major" : "Customer Segment Major",
    "Customer Segment Sub Major" : "Customer Segment Sub Major",
    "Customer Segment Minor" : "Customer Segment Minor",
    "End Market Major" : "End Market Major",
    "End Market Sub Major" : "End Market Sub Major",
    "End Market Minor" : "End Market Minor",
    "Industry" : "Industry",
    "Specialization" : "Specialization",

}



### Definir columnas de Market

In [20]:
cols_final = [
    "llave", "ID",
    "Global Parent", "GP Country", "GP State", "GP City",
    "GP Latitude", "GP Longitude", "GP Super Region",
    "GP Region", "GP Sub Region", "ID Customer",
    "Ship To Number", "Customer Name", "Channel",
    "Ship To Country", "Ship To State", "Ship To City",
    "CN Latitude", "CN Longitude", "CN Continent",
    "CN Super Region", "CN Region", "CN Sub Region",
    "Customer Segment Major", "Customer Segment Sub Major",
    "Customer Segment Minor", "End Market Major",
    "End Market Sub Major", "End Market Minor",
    "Status",  "Industry",
    "Specialization", "ID_Global parent"
]

### Concatenar

In [ ]:
# Renombrar columnas
customer = customer.rename(columns=customer_rename)
potential = potential.rename(columns=potential_rename)

# Asegurar mismas columnas
customer = customer.reindex(columns=cols_final)
potential = potential.reindex(columns=cols_final)

# Concatenar
market = pd.concat([customer, potential], ignore_index=True)

# Market actualizado
market = pd.read_csv(r'C:\Users\Ivan.DeLaFuente\OneDrive - Quaker Houghton\Documents\QH WorkDay\Junio 2026\T&P Market\T&P_Market.csv')

### Market Geo

In [22]:
# Normalizar campos de ubicación
def normalize_text(series):
    
    def clean_value(x):
        # Si es nulo -> mantenerlo
        if pd.isna(x):
            return None
        
        # Convertir a string
        x = str(x).strip().upper()
        
        # ✅ Reemplazar guiones por espacios
        x = x.replace("-", " ")
        
        # Quitar acentos
        try:
            x = unicodedata.normalize('NFKD', x)\
                .encode('ASCII', 'ignore')\
                .decode('utf-8')
        except:
            return None
        
        # Quitar caracteres especiales (ya sin guiones)
        x = re.sub(r'[^A-Z\s]', '', x)
        
        # Limpiar espacios
        x = re.sub(r'\s+', ' ', x).strip()
        
        # Vacíos -> None
        if x == "" or x == "NAN":
            return None
        
        return x

    return series.apply(clean_value)

# Market
market['M GP Country norm'] = normalize_text(market['GP Country'])
market['M GP State norm'] = normalize_text(market['GP State'])
market['M GP City norm'] = normalize_text(market['GP City'])

market['M CN Country norm'] = normalize_text(market['Ship To Country'])
market['M CN State norm'] = normalize_text(market['Ship To State'])
market['M CN City norm'] = normalize_text(market['Ship To City'])

# Regiones
regions["R Country norm"] = normalize_text(regions["Local Company Country"])
regions.rename(columns={
    'Continent':'R Continent', 'Super Region':'R Super Region',
    'Region':'R Region', 'Sub Region':'R Sub Region',
    'Region':'R Region', 'Sub Region':'R Sub Region'}, inplace=True)

# Coordenadas
coordinates["Cor Country norm"] = normalize_text(coordinates["Country"])
coordinates["Cor State norm"]   = normalize_text(coordinates["State"])
coordinates["Cor City norm"]    = normalize_text(coordinates["City"])
coordinates.rename(columns={"Latitude": "Cor Latitude", "Longitude": "Cor Longitude"}, inplace=True)

# Eliminar duplicados de coordenadas
coordinates = coordinates.drop_duplicates(
    subset=["Cor Country norm", "Cor State norm", "Cor City norm"]
)

### Remplazos en States y Cities

In [23]:
reemplazos_states = {
    "STOCKHOLM COUNTY": "STOCKHOLM",
    "COUNTY OFFALY": "OFFALY",
    "CASTILE AND LEON": "CASTILLE AND LEON",
    "BEIJING": "PEKIN",
    "LOWER SILESIAN VOIVODESHIP": "LOWER SILESIAN",
    "NORDRHEIN WESTFALEN": "NORTH RHINE WESTPHALIA",
    "KITWE": "COPPERBELT",
    "MORAVIAN SILESIAN REGION": "MORAVIAN SILESIAN",
    "SILESIAN VOIVODESHIP": "SILESIAN",
    "BADEN WÜRTTEMBERG": "BADEN WURTTEMBERG",

    # Oman
    "AL BATINAH": "AL BATINAH NORTH GOVERNORATE",

    # Spain
    "PAIS VASCO": "BASQUE COUNTRY",
    "REGION METROPOLITANA": "METROPOLITAN REGION",

    "ILEDEFRANCE": "ILE DE FRANCE",
    "BOURGOGNE FRANCHE COMTE": "BOURGOGNE FRANCHE COMTE",  # consistency
    "MORAVIA SILESIA": "MORAVIA SILESIA",
    "BADEN WURTTEMBERG": "BADEN WURTTEMBERG",  # asegurado

    "IMP": "MADHYA PRADESH",
    "AL KHOBAR" : "EASTERN PROVINCE",
    "MINSK REGION": "MINSK",
    "BENI SUEF GOVERNORATE": "BENI SUEF",
    "MEDINA": "AL MADINAH",
    "CAIRO GOVERNORATE" : "CAIRO",
    "GIZA GOVERNORATE" : "GIZA",
    "CENTRAL BOHEMIA" : "CENTRAL BOHEMIAN",
    "OREBRO COUNTY" : "OREBRO",
    "CASTILLE AND LEON" : "CASTILE AND LEON",
    "BOURGOGNE FRANCHE COMTE" : "BOURGOGNE FRANCHE COMTE",
    "MORAVIA SILESIA" : "MORAVIAN SILESIAN",
    "SUEZ" : "SUEZ GOVERNORATE",
    "MOSCOW" : "MOSCOW OBLAST",
    "SINGAPORE" : "CENTRAL SINGAPORE"
    
}

market["M GP State norm"] = market["M GP State norm"].replace(reemplazos_states)
market["M CN State norm"] = market["M CN State norm"].replace(reemplazos_states)

In [24]:
reemplazos_cities = {
    
    "ST. LOUIS": "SAINT LOUIS",
    "ST. CLAIRSVILLE": "ST CLAIRSVILLE",
    "WITBANK": "EMALAHLENI",
    "GRÄFELFING": "GRAFELFING",
    "KOCAELI": "0",
    "SUNDERN": "DUISBURG",
    "PARKLANDS": "KITWE",
    "GIJÓN": "GIJON",
    "XI’AN": "XIAN",


    # Counties → canonical city
    "FAYETTE COUNTY": "FAYETTE",
    "KANAWHA COUNTY": "KANAWHA",
    "WYOMING COUNTY": "WYOMING",
    "HUMBOLDT COUNTY": "HUMBOLDT",
    "LANDER COUNTY": "LANDER",
    "BOONE COUNTY": "BOONE",
    "FLOYD COUNTY": "FLOYD",
    "WAYNE COUNTY": "WAYNE",

    "XUCHAGN": "XUCHANG",
    "WANGIN": "WAGIN",
    "SAINTGEORGES": "SAINT GEORGES",
    "VIA TIERI": "TIERI",
    "ALTZO": "ALTSASU",
    "EL CAIRO" : "CAIRO",
    "WILLIAMSVILE" : "WILLIAMSVILLE",
    "LARGS NORTHS" : "LARGS NORTH",
    "TH OF OCTOBER" : "6TH OF OCTOBER",
    "ROZELL" : "ROZELLE",
    "BIELSKO BIAA" :  "BIELSKO BIALA",
    "MT HOPE" : "MOUNT HOPE",
    "MAHD AD DAHAB" : "MAHD ADH DHAHAB"

}

market["M GP City norm"] = market["M GP City norm"].replace(reemplazos_cities)
market["M CN City norm"] = market["M CN City norm"].replace(reemplazos_cities)

### Merge de coordinates con 3 llaves

In [25]:
# Global Parent
market = market.merge(coordinates[[
    "Cor Country norm", "Cor State norm", "Cor City norm",
    "Cor Latitude", "Cor Longitude"
]],
left_on=["M GP Country norm", "M GP State norm", "M GP City norm"],
right_on=["Cor Country norm", "Cor State norm", "Cor City norm"],
how="left")

# Actualizar columnas existentes
market["GP Latitude"] = market["Cor Latitude"].fillna(market["GP Latitude"])
market["GP Longitude"] = market["Cor Longitude"].fillna(market["GP Longitude"])

# Eliminar columnas auxiliares
market = market.drop(columns=["Cor Country norm", "Cor State norm", 
        "Cor City norm", "Cor Latitude", "Cor Longitude"])

In [26]:
# Company Name
market = market.merge(coordinates[[
    "Cor Country norm", "Cor State norm", "Cor City norm",
    "Cor Latitude", "Cor Longitude"
]],
left_on=["M CN Country norm", "M CN State norm", "M CN City norm"],
right_on=["Cor Country norm", "Cor State norm", "Cor City norm"],
how="left")

# Actualizar columnas existentes
market["CN Latitude"] = market["Cor Latitude"].fillna(market["CN Latitude"])
market["CN Longitude"] = market["Cor Longitude"].fillna(market["CN Longitude"])

# Eliminar columnas auxiliares
market = market.drop(columns=["Cor Country norm", "Cor State norm", 
        "Cor City norm", "Cor Latitude", "Cor Longitude"])

### Merge de regiones

In [27]:
# Global Parent
market = market.merge(regions[[
    "R Country norm", "R Continent", "R Super Region",
    "R Region", "R Sub Region"
]],
left_on="M GP Country norm",
right_on="R Country norm",
how="left"
)

# Actualizar columnas existentes
market["GP Super Region"] = market["R Super Region"].fillna(market["GP Super Region"])
market["GP Region"] = market["R Region"].fillna(market["GP Region"])
market["GP Sub Region"] = market["R Sub Region"].fillna(market["GP Sub Region"])

# Eliminar columnas auxiliares
market = market.drop(columns=["R Country norm", "R Continent", 
    "R Super Region", "R Region", "R Sub Region"])

In [28]:
# Company Name
market = market.merge(regions[[
    "R Country norm", "R Continent", "R Super Region",
    "R Region", "R Sub Region"
]],
left_on="M CN Country norm",
right_on="R Country norm",
how="left"
)

# Actualizar columnas existentes
market["CN Continent"] = market["R Continent"].fillna(market["CN Continent"])
market["CN Super Region"] = market["R Super Region"].fillna(market["CN Super Region"])
market["CN Region"] = market["R Region"].fillna(market["CN Region"])
market["CN Sub Region"] = market["R Sub Region"].fillna(market["CN Sub Region"])

# Eliminar columnas auxiliares
market = market.drop(columns=["R Country norm", "R Continent",
     "R Super Region", "R Region", "R Sub Region"])

### Visualización de registros sin coordenadas

In [29]:
# Detectar Global Parent sin coordenadas en Market
GP_sin_coordenadas = market[["M GP Country norm", "M GP State norm", "M GP City norm", "GP Latitude", "GP Longitude"]]
GP_sin_coordenadas = GP_sin_coordenadas[
    GP_sin_coordenadas["GP Latitude"].isna() |
    GP_sin_coordenadas["GP Longitude"].isna()
].drop_duplicates()

In [30]:
# Detectar Company names sin coordenadas en Market
CN_sin_coordenadas = market[["M CN Country norm", "M CN State norm", "M CN City norm", "CN Latitude", "CN Longitude"]]
CN_sin_coordenadas = CN_sin_coordenadas[
    CN_sin_coordenadas["CN Latitude"].isna() |
    CN_sin_coordenadas["CN Longitude"].isna()
].drop_duplicates()

### Cargar nuevas registros de ubicación limpios y analizar existentes y faltantes
Los nuevos registros salieron de exportar CN_sin_coordinates y limpiar esas filas usando un LLM

In [31]:
# Visualización de registros que estan en el original
new_cord = pd.read_csv(r'C:\Users\Ivan.DeLaFuente\OneDrive - Quaker Houghton\Documents\QH WorkDay\Junio 2026\T&P Market\New_Coordinates.csv')
new_cord = new_cord.drop_duplicates()

new_cord["N Country norm"] = normalize_text(new_cord["Limpia 1"])
new_cord["N State norm"]   = normalize_text(new_cord["Limpia 2"])
new_cord["N City norm"]    = normalize_text(new_cord["Limpia 3"])


merged = new_cord.merge(
    coordinates[["Cor Country norm", "Cor State norm", "Cor City norm"]],
    left_on=["N Country norm", "N State norm", "N City norm"],
    right_on=["Cor Country norm", "Cor State norm", "Cor City norm"],
    how="left",
    indicator=True
)

missing = merged[merged["_merge"] == "left_only"]
missing = missing[["Limpia 1", "Limpia 2", "Limpia 3", "Latitud Limpia", "Longitud Limpia"]]
existing = merged[merged["_merge"] == "both"]
existing = existing[["Limpia 1", "Limpia 2", "Limpia 3", "Latitud Limpia", "Longitud Limpia"]]


### Exportar Market Final a csv

In [32]:
market.drop(columns=["M GP Country norm", "M GP State norm", "M GP City norm", 
     "M CN Country norm", "M CN State norm", "M CN City norm"], inplace=True)
market.to_csv('T&P_Market_Final.csv', index=False)